env1: amortized_lda spectra topicvi harmony

env2: others

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import sys
import warnings
import torch
# RLIB = os.path.join(RHOME,'library')
RHOME = "/home/u12219016/.conda/envs/scgpt/lib/R"
RLIB = os.path.join(RHOME,'library')
os.environ['R_HOME'] = RHOME
os.environ['R_LIB'] = RLIB
# To avoid error: cumsum_cuda_kernel does not have a deterministic implementation
# os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
# os.environ['CUBLAS_WORKSPACE_CONFIG']=':16:8'

warnings.filterwarnings('ignore')
warnings.simplefilter("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module='pandas')
warnings.filterwarnings("ignore", category=DeprecationWarning, module='torchmetrics')
warnings.filterwarnings("ignore", category=DeprecationWarning, module='scarches')

from topicvi import *
sys.path.append('../external/')

from running import RunningPipeline
from running.run_factors import *
from running.run_presentation import *
from running.tl import load_config

In [2]:
method_budle1 = [
    pycogaps, 
    expimap,
    pyliger, dcjcomm, schpf, muvi,
    spike_slab_lda, tree_spike_slab_lda, scgpt_zero_shot
]
method_budle2 = [harmony, scvi, scanvi_seed_label, amortized_lda, spectra, topicvi]

In [32]:
## Check for results valid.

for data_id in data_ids:
    for method in method_budle2+method_budle1:
        if not os.path.exists(
            os.path.join(base_dir, data_id, method.__name__, 'results.npz')
        ):
            print(f"{data_id} {method.__name__}")

GSE262072_CART_Processed_compress, spectra
GSE262072_CART_Processed_compress, pycogaps
GSE262072_CART_Processed_compress, pyliger
GSE262072_CART_Processed_compress, dcjcomm
GSE262072_CART_Processed_compress, muvi
zheng68k, harmony
zheng68k, pyliger
zheng68k, dcjcomm
zheng68k, muvi
Human_pancreas, pyliger
Mariana_24NC, spectra
Mariana_24NC, pycogaps
Mariana_24NC, pyliger
Mariana_24NC, dcjcomm
Mariana_24NC, muvi
zheng68k_sorted, harmony
zheng68k_sorted, pyliger
Lung_atlas, pyliger
Pancan_T_atals_integrated_full, spectra
Pancan_T_atals_integrated_full, pycogaps
Pancan_T_atals_integrated_full, pyliger
Pancan_T_atals_integrated_full, dcjcomm
Pancan_T_atals_integrated_full, muvi
Bassez, spectra
Bassez, pycogaps
Bassez, pyliger
Bassez, dcjcomm
Bassez, muvi


# Subsampled.

In [3]:
base_dir = '../results/subsampled/'
os.listdir(base_dir)

['HLCA_Basal_l3',
 'HLCA_DC_l3',
 'HLCA_Fibroblasts_l3',
 'HLCA_Lymphoid_lf',
 'HLCA_Myeloid_l2',
 'HLCA_Myeloid_lf',
 'HLCA_Secretory_lf',
 'HLCA_T_l3']

In [ ]:
failed_record = []

for data_id in os.listdir(base_dir):    
    
    adata = sc.read_h5ad(os.path.join(base_dir, data_id, 'adata.h5ad'))
    for method in method_budle1:
        print(f"##### {data_id} --- {method.__name__}")
        print(adata.shape)

        config = load_config(os.path.join(base_dir, data_id, 'running_config.yaml'))
        config['extra_kwargs']['topicvi'] = {
            'data_kwargs': dict(label_key=None),
            'train_kwargs': dict(
                pretrain_model = config['extra_kwargs']['topicvi']['train_kwargs']['pretrain_model'],
            ),
            'model_kwargs': dict(
                topic_decoder_params = dict(
                    n_topics_without_prior = 5
                ),
                cluster_decoder_params = dict(
                    center_penalty_weight = 5
                )
            )
        }
        config['train_kwargs']['max_epochs'] = 1000
        config['extra_kwargs']['topicvi_seed_labels'] = config['extra_kwargs']['topicvi']

        try:
            rp = RunningPipeline(method, adata, config)
            rp(verbose=False, save_model=True, check_runned=False)
        except Exception as e:
            failed_record.append([data_id, method.__name__, e])
            raise

In [5]:
failed_record

[]

In [5]:
# config = load_config(os.path.join(base_dir, data_id, 'running_config.yaml'))
# data_kwargs = config['data_kwargs']
# model_kwargs = config['model_kwargs']
# train_kwargs = config['train_kwargs']

# Main data

In [3]:
base_dir = '../results/'
data_ids = os.listdir(base_dir)
data_ids.remove('subsampled')
data_ids.remove('.ipynb_checkpoints')
data_ids

['zenodo_7761954',
 'pbmc1',
 'GSE262072_CART_Processed_compress',
 'zheng68k',
 'pbmc10k',
 'Human_pancreas',
 'Mariana_24NC',
 'zheng68k_sorted',
 'Lung_atlas',
 'Pancan_T_atals_integrated_full',
 'Bassez']

In [ ]:
failed_record = []

for data_id in data_ids:    
    
    adata = sc.read_h5ad(os.path.join(base_dir, data_id, 'adata.h5ad'))
    
    for method in method_budle1:
        print(f"##### {data_id} --- {method.__name__}")
        print(adata.shape)
        if adata.shape[0] > 5e4 and method.__name__ in ['pycogaps', 'pyliger', 'dcjcomm', 'muvi']: continue
        torch.use_deterministic_algorithms(False, warn_only=True)
        try:
            rp = RunningPipeline(method, adata, os.path.join(base_dir, data_id, 'running_config.yaml'))
            rp(verbose=False, save_model=True, check_runned=True)
        except Exception as e:
            print(data_id, method.__name__, e)
            failed_record.append([data_id, method.__name__, e])

In [25]:
# data_id = 'Lung_atlas'
# adata = sc.read_h5ad(os.path.join(base_dir, data_id, 'adata.h5ad'))
# config = load_config(os.path.join(base_dir, data_id, 'running_config.yaml'))
# data_kwargs = config['data_kwargs']
# model_kwargs = config['model_kwargs']
# train_kwargs = config['train_kwargs']